In [102]:
from pathlib import Path
from urllib.request import urlopen
import pandas as pd

# Works whether the notebook is opened from the project root or notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CSV_PATH = RAW_DIR / "BRA.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

URL = "https://football-data.co.uk/new/BRA.csv"

# Download only if the file does not exist.
# Delete BRA.csv manually when you want to fetch a fresh copy.
if not CSV_PATH.exists():
    with urlopen(URL, timeout=30) as response:
        CSV_PATH.write_bytes(response.read())

print(f"Project root: {PROJECT_ROOT}")
print(f"Source file: {CSV_PATH}")
print(f"File size: {CSV_PATH.stat().st_size:,} bytes")

Project root: c:\Users\souza\OneDrive\Documentos\py_enviroment\brasileirao-schedule-project
Source file: c:\Users\souza\OneDrive\Documentos\py_enviroment\brasileirao-schedule-project\data\raw\BRA.csv
File size: 646,941 bytes


In [103]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [104]:
USECOLS = ["League", "Season", "Date", "Time", "Home", "Away", "HG", "AG", "Res"]
DTYPES = {
    "League": "string",
    "Season": "Int64",
    "Time": "string",
    "Home": "string",
    "Away": "string",
    "HG": "Int64",
    "AG": "Int64",
    "Res": "string",
}

df = pd.read_csv(
    CSV_PATH,
    usecols=USECOLS,
    dtype=DTYPES,
    encoding="cp1252",
)

df.columns = df.columns.str.strip().str.lower()
df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")

print(f"Rows: {len(df):,}")
print(f"Date range: {df['date'].min()} — {df['date'].max()}")
print(f"Rows with invalid/missing dates: {df['date'].isna().sum()}")
display(df.head())

Rows: 5,629
Date range: 2012-05-19 00:00:00 — 2026-10-20 00:00:00
Rows with invalid/missing dates: 0


,league,season,date,time,home,away,hg,ag,res
0,Serie A,2012,2012-05-19,22:30,Palmeiras,Portuguesa,1,1,D
1,Serie A,2012,2012-05-19,22:30,Sport Recife,Flamengo RJ,1,1,D
2,Serie A,2012,2012-05-20,01:00,Figueirense,Nautico,2,1,H
3,Serie A,2012,2012-05-20,20:00,Botafogo RJ,Sao Paulo,4,2,H
4,Serie A,2012,2012-05-20,20:00,Corinthians,Fluminense,0,1,A


In [105]:
display(df)

,league,season,date,time,home,away,hg,ag,res
0,Serie A,2012,2012-05-19,22:30,Palmeiras,Portuguesa,1,1,D
1,Serie A,2012,2012-05-19,22:30,Sport Recife,Flamengo RJ,1,1,D
2,Serie A,2012,2012-05-20,01:00,Figueirense,Nautico,2,1,H
3,Serie A,2012,2012-05-20,20:00,Botafogo RJ,Sao Paulo,4,2,H
4,Serie A,2012,2012-05-20,20:00,Corinthians,Fluminense,0,1,A
...,...,...,...,...,...,...,...,...,...
5624,Serie A,2026,2026-10-18,20:00,Fluminense,Santos,1,2,A
5625,Serie A,2026,2026-10-18,20:00,Gremio,Cruzeiro,2,4,A
5626,Serie A,2026,2026-10-18,22:30,Bahia,Flamengo RJ,0,3,A
5627,Serie A,2026,2026-10-18,23:30,Remo,Bragantino,1,1,D


In [106]:
df_2026 = df.query(
    'season == 2026 and (home.str.contains("Flamengo", case=False, na=False) or away.str.contains("Flamengo", case=False, na=False))'
).reset_index(drop=True)
is_2026 = df["season"].eq(2026)
is_flamengo = (
    df["home"].str.contains("Flamengo", case=False, na=False)
    | df["away"].str.contains("Flamengo", case=False, na=False)
)

df_2026_flamengo = (
    df.loc[is_2026 & is_flamengo]
    .sort_values("date")
    .reset_index(drop=True)
)

display(df_2026_flamengo)
df_2026 = df_2026.sort_values(["season", "date"]).reset_index(drop=True)

,league,season,date,time,home,away,hg,ag,res
0,Serie A,2026,2026-01-29,00:30,Sao Paulo,Flamengo RJ,2,1,H
1,Serie A,2026,2026-02-04,22:00,Flamengo RJ,Internacional,1,1,D
2,Serie A,2026,2026-02-11,00:30,Vitoria,Flamengo RJ,1,2,A
3,Serie A,2026,2026-03-12,00:30,Flamengo RJ,Cruzeiro,2,0,H
4,Serie A,2026,2026-03-14,23:30,Botafogo RJ,Flamengo RJ,0,3,A
5,Serie A,2026,2026-03-19,23:00,Flamengo RJ,Remo,3,0,H
6,Serie A,2026,2026-03-22,23:30,Corinthians,Flamengo RJ,1,1,D
7,Serie A,2026,2026-04-03,01:30,Bragantino,Flamengo RJ,3,0,H
8,Serie A,2026,2026-04-05,21:30,Flamengo RJ,Santos,3,1,H
9,Serie A,2026,2026-04-12,22:00,Fluminense,Flamengo RJ,1,2,A


In [107]:
club_cities = {
    "Athletico-PR": ("Curitiba", "PR"),
    "Atletico-MG": ("Belo Horizonte", "MG"),
    "Bahia": ("Salvador", "BA"),
    "Botafogo RJ": ("Rio de Janeiro", "RJ"),
    "Chapecoense-SC": ("Chapecó", "SC"),
    "Corinthians": ("São Paulo", "SP"),
    "Coritiba": ("Curitiba", "PR"),
    "Cruzeiro": ("Belo Horizonte", "MG"),
    "Flamengo RJ": ("Rio de Janeiro", "RJ"),
    "Fluminense": ("Rio de Janeiro", "RJ"),
    "Gremio": ("Porto Alegre", "RS"),
    "Internacional": ("Porto Alegre", "RS"),
    "Mirassol": ("Mirassol", "SP"),
    "Palmeiras": ("São Paulo", "SP"),
    "Bragantino": ("Bragança Paulista", "SP"),
    "Remo": ("Belém", "PA"),
    "Santos": ("Santos", "SP"),
    "Sao Paulo": ("São Paulo", "SP"),
    "Vasco": ("Rio de Janeiro", "RJ"),
    "Vitoria": ("Salvador", "BA"),
}

club_locations = pd.DataFrame(
    [
        {"club": club, "city": city, "state": state}
        for club, (city, state) in club_cities.items()
    ]
)

clubs_in_data = set(df["home"].dropna()) | set(df["away"].dropna())
unmapped_clubs = sorted(clubs_in_data - set(club_locations["club"]))

print("Clubs in data without a city mapping:", unmapped_clubs)
display(club_locations)

club_locations.to_csv(PROCESSED_DIR / "cities.csv", index=False)

Clubs in data without a city mapping: ['America MG', 'Atletico GO', 'Avai', 'CSA', 'Ceara', 'Criciuma', 'Cuiaba', 'Figueirense', 'Fortaleza', 'Goias', 'Joinville', 'Juventude', 'Nautico', 'Parana', 'Ponte Preta', 'Portuguesa', 'Santa Cruz', 'Sport Recife']


,club,city,state
0,Athletico-PR,Curitiba,PR
1,Atletico-MG,Belo Horizonte,MG
2,Bahia,Salvador,BA
3,Botafogo RJ,Rio de Janeiro,RJ
4,Chapecoense-SC,Chapecó,SC
5,Corinthians,São Paulo,SP
6,Coritiba,Curitiba,PR
7,Cruzeiro,Belo Horizonte,MG
8,Flamengo RJ,Rio de Janeiro,RJ
9,Fluminense,Rio de Janeiro,RJ


In [108]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

CITIES_PATH = PROCESSED_DIR / "cities.csv"
COORDINATES_PATH = PROCESSED_DIR / "city_coordinates.csv"

club_locations = pd.read_csv(CITIES_PATH, dtype={"club": "string", "city": "string", "state": "string"})
unique_cities = club_locations[["city", "state"]].drop_duplicates().reset_index(drop=True)

# Load cached coordinates, if available.
if COORDINATES_PATH.exists():
    cached = pd.read_csv(COORDINATES_PATH)
    cached = cached[["city", "state", "latitude", "longitude"]].drop_duplicates(["city", "state"])
else:
    cached = pd.DataFrame(columns=["city", "state", "latitude", "longitude"])

locations = unique_cities.merge(
    cached,
    on=["city", "state"],
    how="left",
    validate="one_to_one",
)

missing_coords = locations["latitude"].isna() | locations["longitude"].isna()

if missing_coords.any():
    geolocator = Nominatim(user_agent="brasileirao_schedule_project")
    geocode = RateLimiter(
        geolocator.geocode,
        min_delay_seconds=1.1,
        max_retries=2,
        swallow_exceptions=True,
    )

    for idx in locations.index[missing_coords]:
        city = locations.at[idx, "city"]
        state = locations.at[idx, "state"]
        result = geocode(f"{city}, {state}, Brazil")

        if result is not None:
            locations.at[idx, "latitude"] = result.latitude
            locations.at[idx, "longitude"] = result.longitude

locations.to_csv(COORDINATES_PATH, index=False)

unresolved = locations[locations["latitude"].isna() | locations["longitude"].isna()]
print(f"Cities: {len(locations)}")
print(f"Cities without coordinates: {len(unresolved)}")
display(unresolved if not unresolved.empty else locations)

Cities: 11
Cities without coordinates: 0


,city,state,latitude,longitude
0,Curitiba,PR,-25.429596,-49.271272
1,Belo Horizonte,MG,-19.922732,-43.945095
2,Salvador,BA,-12.982250,-38.481277
3,Rio de Janeiro,RJ,-22.911014,-43.209373
4,Chapecó,SC,-27.111047,-52.595898
5,São Paulo,SP,-23.550651,-46.633382
6,Porto Alegre,RS,-30.032500,-51.230377
7,Mirassol,SP,-20.814141,-49.507449
8,Bragança Paulista,SP,-22.952023,-46.541859
9,Belém,PA,-1.450560,-48.468245


In [109]:
from itertools import product
from geopy.distance import geodesic

# Reload saved coordinates and validate them.
locations = pd.read_csv(COORDINATES_PATH)
locations = locations.drop_duplicates(["city", "state"]).reset_index(drop=True)

required = {"city", "state", "latitude", "longitude"}
missing = required - set(locations.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

locations["latitude"] = pd.to_numeric(locations["latitude"], errors="coerce")
locations["longitude"] = pd.to_numeric(locations["longitude"], errors="coerce")

unresolved = locations[locations[["latitude", "longitude"]].isna().any(axis=1)]
if not unresolved.empty:
    raise ValueError(
        "Some cities have no coordinates. Review city_coordinates.csv:\n"
        + unresolved[["city", "state"]].to_string(index=False)
    )

rows = []
records = list(locations.itertuples(index=False))

for origin, destination in product(records, repeat=2):
    km = geodesic(
        (origin.latitude, origin.longitude),
        (destination.latitude, destination.longitude),
    ).km

    rows.append({
        "origin_city": origin.city,
        "origin_state": origin.state,
        "destination_city": destination.city,
        "destination_state": destination.state,
        "distance_km": round(km, 2),
    })

distance_matrix = pd.DataFrame(rows)
DISTANCES_PATH = PROCESSED_DIR / "city_distances.csv"
distance_matrix.to_csv(DISTANCES_PATH, index=False)

print(f"Unique cities: {len(locations)}")
print(f"Origin-destination pairs: {len(distance_matrix)}")
print(f"Saved to: {DISTANCES_PATH}")
display(distance_matrix.head())

Unique cities: 11
Origin-destination pairs: 121
Saved to: c:\Users\souza\OneDrive\Documentos\py_enviroment\brasileirao-schedule-project\data\processed\city_distances.csv


,origin_city,origin_state,destination_city,destination_state,distance_km
0,Curitiba,PR,Curitiba,PR,0.00
1,Curitiba,PR,Belo Horizonte,MG,819.21
2,Curitiba,PR,Salvador,BA,1782.92
3,Curitiba,PR,Rio de Janeiro,RJ,676.10
4,Curitiba,PR,Chapecó,SC,380.75


In [110]:
locations.to_csv(
    PROJECT_ROOT / "data" / "processed" / "city_coordinates.csv",
    index=False,
)

In [111]:

# Calculate estimated travel distance for every club in the 2026 season

import pandas as pd

SEASON = 2026
DISTANCES_PATH = PROCESSED_DIR / "city_distances.csv"

city_distances = pd.read_csv(DISTANCES_PATH)

# 1. Map each club's home city
club_locations_clean = club_locations.drop_duplicates("club")

home_locations = club_locations_clean.rename(
    columns={
        "club": "home",
        "city": "match_city",
        "state": "match_state",
    }
)

away_locations = club_locations_clean.rename(
    columns={
        "club": "away",
        "city": "origin_city",
        "state": "origin_state",
    }
)

# 2. Select completed matches from the chosen season
matches = df.loc[
    df["season"].eq(SEASON)
    & df["hg"].notna()
    & df["ag"].notna()
].copy()

# 3. Attach the home team's city and the away team's home city
matches = (
    matches
    .merge(
        home_locations,
        on="home",
        how="left",
        validate="many_to_one",
    )
    .merge(
        away_locations,
        on="away",
        how="left",
        validate="many_to_one",
    )
)

# Validate club mappings
missing_mapping = matches[
    matches["match_city"].isna()
    | matches["origin_city"].isna()
]

if not missing_mapping.empty:
    raise ValueError(
        "Some clubs are missing city mappings:\n"
        + missing_mapping[["home", "away"]]
        .drop_duplicates()
        .to_string(index=False)
    )

# 4. Look up one-way distance from the away club's city
#    to the home team's city
distance_lookup = city_distances[
    [
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state",
        "distance_km",
    ]
].drop_duplicates(
    [
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state",
    ]
)

matches = matches.merge(
    distance_lookup,
    left_on=[
        "origin_city",
        "origin_state",
        "match_city",
        "match_state",
    ],
    right_on=[
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state",
    ],
    how="left",
    validate="many_to_one",
)

# 5. Each away club travels to the match and returns home.
#    The home club has no intercity travel for that match.
matches["away_travel_km"] = 2 * matches["distance_km"]

if matches["away_travel_km"].isna().any():
    missing_distances = matches.loc[
        matches["away_travel_km"].isna(),
        ["home", "away", "origin_city", "match_city"],
    ].drop_duplicates()

    raise ValueError(
        "Missing distances for these matches:\n"
        + missing_distances.to_string(index=False)
    )

# 6. Create one travel record per club per match
away_trips = matches[
    ["date", "season", "away", "home", "match_city", "away_travel_km"]
].rename(
    columns={
        "away": "club",
        "home": "opponent",
        "match_city": "destination_city",
        "away_travel_km": "travel_km",
    }
)

# 7. Sum distance for every club
club_totals = (
    away_trips.groupby("club", as_index=False)
    .agg(
        away_matches=("club", "size"),
        total_travel_km=("travel_km", "sum"),
    )
    .sort_values("total_travel_km", ascending=False)
    .reset_index(drop=True)
)

club_totals["total_travel_km"] = club_totals["total_travel_km"].round(2)

# 8. Total distance traveled by all clubs combined
league_total_km = club_totals["total_travel_km"].sum()

display(club_totals)

print(f"Season: {SEASON}")
print(f"Completed matches considered: {len(matches)}")
print(f"Clubs with away matches: {len(club_totals)}")
print(f"Combined estimated travel: {league_total_km:,.2f} km")


,club,away_matches,total_travel_km
0,Remo,15,73213.10
1,Vitoria,16,44288.34
2,Bahia,15,42987.10
3,Gremio,15,36738.70
4,Internacional,16,36269.54
5,Chapecoense-SC,16,29877.46
6,Cruzeiro,16,24719.96
7,Mirassol,15,24174.02
8,Atletico-MG,15,23311.28
9,Coritiba,16,23252.76


Season: 2026
Completed matches considered: 309
Clubs with away matches: 20
Combined estimated travel: 551,919.94 km
